# Encoding Molecules with Learned Embedding
We will provide module codes to encode molecules with the feature engineering techniques we learned in this lecture.

In [1]:
import torch
import torch.nn as nn

In [3]:
molecules = {
    "water": "O",
    "methane": "C",
    "ethanol": "CCO",
    "benzene": "c1ccccc1"
}

## Build the vocabulary
We will use the following syntax to concatenate all strings together:
`''.join(a_list_of_strings)`

Then use `set(joined_string)` removes repeated elements in the string. (a set cannot have repeated elements).

In [6]:
joined_string = ''.join(molecules.values())
print(joined_string)
vocab = set(joined_string)
print(vocab)

OCCCOc1ccccc1
{'c', '1', 'C', 'O'}


### Build a look up dictionary: tokens - indices

In [7]:
char2idx = {c: i for i, c in enumerate(vocab)}
idx2char = {i: c for c, i in char2idx.items()}
print(char2idx)

{'c': 0, '1': 1, 'C': 2, 'O': 3}


In [8]:
vocab_size = len(vocab)
max_len = max(len(smi) for smi in molecules.values()) # use the longest SMILES length as the maximum

In [10]:
# Convert SMILES to index tensor
def smiles_to_indices(smiles, max_len=max_len):
    indices = [char2idx[c] for c in smiles]
    # # pad with 0 if shorter than max_len
    # indices += [0] * (max_len - len(indices))
    return torch.tensor(indices, dtype=torch.long)
print(smiles_to_indices(molecules["water"]))

tensor([3])


In [11]:
# Define an Embedding Network
class SmilesEmbedder(nn.Module):
    def __init__(self, vocab_size, embed_dim=16):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)

    def forward(self, x):
        # x is a one-dimensional vector storing token indices
        x_emb = self.embedding(x)       #  (seq_len, embed_dim)
        mol_embed = x_emb.mean(dim=0)   # simple mean pooling
        # nn.Linear(embed_dim, hidden_dim)
        # nn.ReLU()
        # nn.Linear(hidden_dim, output_dim)
        return mol_embed


In [12]:
# initialize model
embedder = SmilesEmbedder(vocab_size=vocab_size, embed_dim=10)

# generate learned embeddings
embeddings = {}
for name, smi in molecules.items():
    idx_tensor = smiles_to_indices(smi)
    embeddings[name] = embedder(idx_tensor)

# Print embeddings
for name, emb in embeddings.items():
    print(f"{name} embedding: {emb.detach().numpy()}")

water embedding: [-0.95503014 -0.8366163   1.131706    0.24066158 -0.09375531  1.1068242
 -0.7373721  -1.6373614  -0.15548782 -1.3916769 ]
methane embedding: [ 0.52441573 -0.98141104  0.4666776  -0.10965981 -0.45961595  3.351767
  0.4613206   0.08355197 -1.1305171   0.11524805]
ethanol embedding: [ 0.03126711 -0.9331462   0.6883537   0.00711398 -0.33766243  2.6034527
  0.06175637 -0.49008584 -0.80550736 -0.38706028]
benzene embedding: [ 0.42260644 -0.68698466  0.35048404 -0.18271166 -1.5678761   0.73803663
  0.738629    1.1137303   0.63782215 -0.15164216]
